# Lung Cancar Prediction: Data Pre-Processing

| Variable Name | Description |
| --- | --- |
| **age** | Age of the individual in years |
| **gender** | Biological sex (0 = Female, 1 = Male) |
| **education_years** | Total years of formal education completed |
| **income_level** | Socioeconomic status on an ordinal scale (1 = lowest, 5 = highest) |
| **smoker** | Indicates whether the individual has a history of smoking (0 = No, 1 = Yes) |
| **smoking_years** | Total number of years the individual has smoked |
| **cigarettes_per_day** | Average number of cigarettes smoked per day |
| **pack_years** | Cumulative smoking exposure combining duration and intensity |
| **passive_smoking** | Exposure to secondhand smoke (0 = No, 1 = Yes) |
| **air_pollution_index** | Air quality index representing long-term pollution exposure |
| **occupational_exposure** | Exposure to hazardous substances at work (0 = No, 1 = Yes) |
| **radon_exposure** | History of radon exposure (0 = No, 1 = Yes) |
| **family_history_cancer** | Family history of cancer (0 = No, 1 = Yes) |
| **copd** | Diagnosis of chronic obstructive pulmonary disease (0 = No, 1 = Yes) |
| **asthma** | History of asthma (0 = No, 1 = Yes) |
| **previous_tb** | History of tuberculosis infection (0 = No, 1 = Yes) |
| **chronic_cough** | Presence of long-term cough symptoms (0 = No, 1 = Yes) |
| **chest_pain** | Reports of chest pain (0 = No, 1 = Yes) |
| **shortness_of_breath** | Presence of breathing difficulty (0 = No, 1 = Yes) |
| **fatigue** | Persistent fatigue symptoms (0 = No, 1 = Yes) |
| **bmi** | Body mass index category value |
| **oxygen_saturation** | Blood oxygen saturation level (%) |
| **fev1_x10** | Lung function measure derived from forced expiratory volume |
| **crp_level** | C-reactive protein level indicating inflammation |
| **xray_abnormal** | Abnormal findings in chest imaging (0 = No, 1 = Yes) |
| **exercise_hours_per_week** | Average weekly physical activity duration |
| **diet_quality** | Overall dietary quality score (1 = poor, 5 = excellent) |
| **alcohol_units_per_week** | Average alcohol consumption per week |
| **healthcare_access** | Access to healthcare services (1 = poor, 5 = excellent) |
| **lung_cancer_risk** | **(Target)** Indicator of elevated lung cancer risk (0 = No, 1 = Yes) |

---

### Pre-Processing Approach Based On EDA 
Handling Multicollinearity: The Heatmap showed pack_years, smoking_years, and cigarettes_per_day are almost identical in information. We will drop smoking_years and cigarettes_per_day and keep only pack_years (the strongest predictor) to prevent model confusion.

Feature Engineering: The Interaction Plot showed that pollution specifically hurts smokers. We will create a new feature: pollution_smoker_interaction.

Symptom Aggregation: Since symptoms like cough and breathlessness often appear together (clustering), we will create a symptom_score to capture overall respiratory distress.

---


### 1. Setup and Initialization

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE # for class imbalance

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('data/lung_cancer.csv')
df.head()

,age,gender,education_years,income_level,smoker,smoking_years,cigarettes_per_day,pack_years,passive_smoking,air_pollution_index,...,bmi,oxygen_saturation,fev1_x10,crp_level,xray_abnormal,exercise_hours_per_week,diet_quality,alcohol_units_per_week,healthcare_access,lung_cancer_risk
0,60,1,20,2,1,16,15,12,0,71,...,20,94,29,6,1,5,4,13,3,1
1,53,0,12,4,0,0,0,0,1,66,...,25,96,35,4,0,5,2,0,3,0
2,62,1,15,3,1,9,29,13,1,69,...,23,95,29,9,1,1,4,2,1,1
3,73,1,12,3,0,0,0,0,0,47,...,18,96,32,0,0,0,3,10,4,0
4,52,1,13,1,0,0,0,0,0,94,...,16,97,36,8,0,6,2,9,2,0


In [3]:
df.shape

(5000, 30)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   age                      5000 non-null   int64
 1   gender                   5000 non-null   int64
 2   education_years          5000 non-null   int64
 3   income_level             5000 non-null   int64
 4   smoker                   5000 non-null   int64
 5   smoking_years            5000 non-null   int64
 6   cigarettes_per_day       5000 non-null   int64
 7   pack_years               5000 non-null   int64
 8   passive_smoking          5000 non-null   int64
 9   air_pollution_index      5000 non-null   int64
 10  occupational_exposure    5000 non-null   int64
 11  radon_exposure           5000 non-null   int64
 12  family_history_cancer    5000 non-null   int64
 13  copd                     5000 non-null   int64
 14  asthma                   5000 non-null   int64
 15  prev

In [5]:
df.describe()

,age,gender,education_years,income_level,smoker,smoking_years,cigarettes_per_day,pack_years,passive_smoking,air_pollution_index,...,bmi,oxygen_saturation,fev1_x10,crp_level,xray_abnormal,exercise_hours_per_week,diet_quality,alcohol_units_per_week,healthcare_access,lung_cancer_risk
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,...,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0000,5000.000000,5000.000000
mean,54.566400,0.488200,11.510400,2.554400,0.454800,8.822200,6.685600,6.249000,0.347600,64.278800,...,23.604800,96.089200,31.475800,4.593600,0.201400,2.580400,2.534800,5.8850,2.536800,0.248800
std,11.932731,0.499911,2.953987,0.982667,0.498003,11.651598,9.032888,9.964901,0.476256,19.479809,...,3.937222,3.412385,5.213679,5.333566,0.401086,1.834178,0.985385,4.4239,0.984093,0.432361
min,18.000000,0.000000,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000,...,16.000000,85.000000,5.000000,0.000000,0.000000,0.000000,1.000000,0.0000,1.000000,0.000000
25%,47.000000,0.000000,10.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,51.000000,...,21.000000,95.000000,30.000000,1.000000,0.000000,1.000000,2.000000,2.0000,2.000000,0.000000
50%,55.000000,0.000000,11.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,64.000000,...,24.000000,97.000000,33.000000,3.000000,0.000000,2.000000,3.000000,6.0000,3.000000,0.000000
75%,62.000000,1.000000,13.250000,3.000000,1.000000,18.000000,14.000000,10.000000,1.000000,78.000000,...,26.000000,99.000000,35.000000,7.000000,0.000000,4.000000,3.000000,9.0000,3.000000,0.000000
max,90.000000,1.000000,20.000000,5.000000,1.000000,52.000000,44.000000,60.000000,1.000000,130.000000,...,37.000000,100.000000,37.000000,33.000000,1.000000,10.000000,5.000000,23.0000,5.000000,1.000000


---
### 2. Logical Data Cleaning
As seen in the "Sanity Check" scatter plot, we must ensure non-smokers have 0 values for all smoking metrics.

In [6]:
# 1. Enforce Logical Consistency for Non-Smokers
# If smoker == 0, all smoking stats must be 0
mask_non_smoker = df['smoker'] == 0
smoking_cols = ['cigarettes_per_day', 'smoking_years', 'pack_years']

df.loc[mask_non_smoker, smoking_cols] = 0

---
### 3. Feature Engineering (Based on EDA)

In [7]:
# A. Interaction Feature: Pollution * Smoker
# EDA showed pollution mainly increases risk for smokers.
df['pollution_smoker_interaction'] = df['air_pollution_index'] * df['smoker']

# B. Symptom Score
# Aggregating binary symptoms into a single 'Respiratory Health Score'
symptoms = ['chronic_cough', 'shortness_of_breath', 'chest_pain', 'fatigue', 'wheezing', 'swallowing_difficulty', 'clubbing_of_finger_nails'] 
# Note: Adjust the list based on actual columns in your df. 
# Based on your dataset columns:
symptoms_available = ['chronic_cough', 'shortness_of_breath', 'chest_pain', 'fatigue']
df['symptom_score'] = df[symptoms_available].sum(axis=1)

print("New Features Created: 'pollution_smoker_interaction', 'symptom_score'")

New Features Created: 'pollution_smoker_interaction', 'symptom_score'


---
### 4. Feature Selection (Reducing Multicollinearity)
The heatmap revealed pack_years correlation is 0.83, while smoking_years is 0.72. Since pack_years is a derived metric of intensity and duration, keeping the others creates noise.

In [8]:
# Drop redundant smoking columns to reduce multicollinearity
# We keep 'pack_years' (most information) and 'smoker' (categorical status)
cols_to_drop = ['cigarettes_per_day', 'smoking_years']
df_reduced = df.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"New shape: {df_reduced.shape}")

Dropped columns: ['cigarettes_per_day', 'smoking_years']
New shape: (5000, 30)


---
### 5. Defining X and y
We separate the target variable and define which columns are numerical for scaling.

In [9]:
X = df_reduced.drop('lung_cancer_risk', axis=1)
y = df_reduced['lung_cancer_risk']

# Identify Numerical Columns for Scaling
# Note: Binary cols (0/1) usually don't need standard scaling
numerical_cols = [
    'age', 'education_years', 'income_level', 'pack_years', 
    'air_pollution_index', 'bmi', 'oxygen_saturation', 'fev1_x10', 
    'crp_level', 'exercise_hours_per_week', 'diet_quality', 
    'alcohol_units_per_week', 'healthcare_access', 
    'pollution_smoker_interaction', 'symptom_score'
]

# Identify Binary columns (everything else)
binary_cols = [col for col in X.columns if col not in numerical_cols]

---
### 6. Stratified Train-Test Split
The pie chart showed a 75/25 imbalance. We use stratify=y to maintain this ratio in our test set.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Crucial for imbalanced data
)

print(f"Train samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Train samples: 4000
Test samples: 1000


---
### 7. Feature Scaling (Standardization)

The boxplots showed pack_years and alcohol_units have outliers. Standardization ($z = \frac{x - \mu}{\sigma}$) handles this better than Normalization (Min-Max) for algorithms like Logistic Regression and SVM.

In [11]:
scaler = StandardScaler()

# Fit scaler ONLY on Training data to prevent data leakage
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Transform Test data using the Train scaler
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

---

### 8. Handling Imbalance (SMOTE)
The dataset is imbalanced (25% positive class). We apply SMOTE only to the training set to synthesize new minority samples.

In [12]:
print("Target distribution before SMOTE (Train):")
print(y_train.value_counts(normalize=True))

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nTarget distribution after SMOTE (Train):")
print(y_train_balanced.value_counts(normalize=True))
print(f"Final Training Shape: {X_train_balanced.shape}")

Target distribution before SMOTE (Train):
lung_cancer_risk
0    0.75125
1    0.24875
Name: proportion, dtype: float64

Target distribution after SMOTE (Train):
lung_cancer_risk
0    0.5
1    0.5
Name: proportion, dtype: float64
Final Training Shape: (6010, 29)


---

### 9. Final Data Review
The data is now chemically pure for modeling:

Logically Consistent (No non-smoking smokers).

Engineered (Interaction terms included).

Streamlined (Redundant columns removed).

Scaled (Comparability achieved).

Balanced (No bias towards majority class).

In [13]:
X_train_balanced.head()

,age,gender,education_years,income_level,smoker,pack_years,passive_smoking,air_pollution_index,occupational_exposure,radon_exposure,...,oxygen_saturation,fev1_x10,crp_level,xray_abnormal,exercise_hours_per_week,diet_quality,alcohol_units_per_week,healthcare_access,pollution_smoker_interaction,symptom_score
0,0.206012,0,0.842465,1.478636,0,-0.630094,0,0.286329,0,0,...,1.151166,0.108129,0.065766,0,-0.315340,0.472581,-0.653506,0.462500,-0.856869,-0.030562
1,0.290739,1,0.166533,1.478636,0,-0.630094,0,-1.510203,0,0,...,-0.019685,0.488031,-0.305533,0,-0.315340,0.472581,-0.879242,2.479948,-0.856869,-1.093596
2,-0.980159,0,1.180431,1.478636,0,-0.630094,1,-0.072978,1,0,...,0.565741,0.488031,-0.305533,0,0.773448,-0.541541,-1.330714,0.462500,-0.856869,1.032472
3,2.069996,1,0.166533,-0.554203,0,-0.630094,1,-0.586272,0,0,...,-0.019685,0.488031,-0.676832,0,-0.859734,-1.555663,-1.104978,1.471224,-0.856869,-1.093596
4,0.544919,0,-1.861264,0.462217,0,-0.630094,1,-1.561532,0,1,...,1.151166,0.108129,-0.676832,0,1.862236,0.472581,0.249438,0.462500,-0.856869,-0.030562


In [14]:
X_train_balanced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6010 entries, 0 to 6009
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   age                           6010 non-null   float64
 1   gender                        6010 non-null   int64  
 2   education_years               6010 non-null   float64
 3   income_level                  6010 non-null   float64
 4   smoker                        6010 non-null   int64  
 5   pack_years                    6010 non-null   float64
 6   passive_smoking               6010 non-null   int64  
 7   air_pollution_index           6010 non-null   float64
 8   occupational_exposure         6010 non-null   int64  
 9   radon_exposure                6010 non-null   int64  
 10  family_history_cancer         6010 non-null   int64  
 11  copd                          6010 non-null   int64  
 12  asthma                        6010 non-null   int64  
 13  pre

In [15]:
X_train_balanced.describe()

,age,gender,education_years,income_level,smoker,pack_years,passive_smoking,air_pollution_index,occupational_exposure,radon_exposure,...,oxygen_saturation,fev1_x10,crp_level,xray_abnormal,exercise_hours_per_week,diet_quality,alcohol_units_per_week,healthcare_access,pollution_smoker_interaction,symptom_score
count,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,...,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000,6010.000000
mean,0.041853,0.432779,-0.018978,0.001657,0.641431,0.481096,0.289517,0.025871,0.201997,0.106988,...,-0.440805,-0.455215,0.445347,0.335441,-0.036434,-0.003749,-0.006232,-0.014347,0.348490,0.295980
std,0.950383,0.495502,0.959892,0.959158,0.479620,1.182939,0.453576,0.968771,0.401523,0.309124,...,1.142216,1.173128,1.153668,0.472184,0.955491,0.957954,0.972150,0.959661,0.996433,1.065209
min,-3.098321,0.000000,-2.199230,-1.570622,0.000000,-0.630094,0.000000,-2.280145,0.000000,0.000000,...,-3.239526,-5.020540,-0.862482,0.000000,-1.404128,-1.555663,-1.330714,-1.554948,-0.856869,-1.093596
25%,-0.563988,0.000000,-0.636721,-0.554203,0.000000,-0.630094,0.000000,-0.637602,0.000000,0.000000,...,-1.190536,-1.221526,-0.676832,0.000000,-0.859734,-0.541541,-0.879242,-0.546224,-0.856869,-0.195320
50%,0.036559,0.000000,-0.075100,0.089962,1.000000,0.266553,0.000000,0.029681,0.000000,0.000000,...,-0.160452,-0.143936,0.251416,0.000000,-0.262295,-0.005161,-0.033866,-0.055529,0.612025,-0.030562
75%,0.641725,1.000000,0.504499,0.462217,1.000000,1.262828,1.000000,0.696964,0.000000,0.000000,...,0.565741,0.488031,1.179665,1.000000,0.773448,0.472581,0.700911,0.462500,1.188061,1.032472
max,3.001987,1.000000,2.870262,2.495056,1.000000,5.347555,1.000000,3.212108,1.000000,1.000000,...,1.151166,1.057883,5.263958,1.000000,4.039812,2.500824,3.861216,2.479948,2.800964,3.158540


In [16]:
y_train_balanced.value_counts()

lung_cancer_risk
0    3005
1    3005
Name: count, dtype: int64

--- 
### 10. Exporting Pre-Processed data

In [17]:
# 1. Recombine features (X) and target (y) for the Training set
# This set is Scaled and Balanced (contains SMOTE synthetic data)
train_processed = pd.concat([X_train_balanced, y_train_balanced], axis=1)

# 2. Recombine features (X) and target (y) for the Testing set
# This set is Scaled but Natural (contains ONLY original real data)
test_processed = pd.concat([X_test, y_test], axis=1)

# 3. Export to CSV files
train_processed.to_csv('data/pre_processed_data/train_processed.csv', index=False)
test_processed.to_csv('data/pre_processed_data/test_processed.csv', index=False)

print(f"1. 'train_processed.csv': {train_processed.shape}")
print(f"2. 'test_processed.csv':  {test_processed.shape}")

1. 'train_processed.csv': (6010, 30)
2. 'test_processed.csv':  (1000, 30)


---

**Thank You!!!**